# Implementierung von Iteration

## Agenda

1. Rückblick: Iteration  
2. Details: *Iterables*, *Iteratoren*, `iter` und `next`  
3. Implementierung von Iteratoren mit Klassen  
4. Implementierung von Iteratoren mit *Generatoren* und `yield`

## 1. Wiederholung: Iteration

*Iteration* bezeichnet einfach den Vorgang, die in einem Container gespeicherten Elemente nacheinander zuzugreifen. Die Reihenfolge der Elemente und ob die Iteration vollständig ist, hängt vom Container ab.

In Python führen wir Iterationen typischerweise mit der `for`-Schleife durch.

In [1]:
# e.g., iterating over a list
l = [2**x for x in range(10)]
for n in l:
    print(n)

1
2
4
8
16
32
64
128
256
512


In [2]:
# e.g., iterating over the key-value pairs in a dictionary
d = {x:2**x for x in range(10)}
for k,v in d.items():
    print(k, '=>', v)

0 => 1
1 => 2
2 => 4
3 => 8
4 => 16
5 => 32
6 => 64
7 => 128
8 => 256
9 => 512


## 2. Rückblick: *iterierbare Objekte*, *Iteratoren*, `iter` und `next`

Wir können über alles iterieren, was *iterierbar* ist. Intuitiv gilt: Wenn etwas als Quelle von Elementen in einer `for`-Schleife verwendet werden kann, ist es iterierbar.

Aber wie funktioniert eine `for`-Schleife eigentlich wirklich? (Zeit für eine Wiederholung!)  
Schau dir die [iter()](https://docs.python.org/3/library/functions.html#iter) und [Iterator](https://docs.python.org/3/glossary.html#term-iterator)-Objekte in der Python-Dokumentation an.

In [3]:
a = 'Hi'
itr = iter(a)
type(itr)

str_ascii_iterator

In [4]:
next(itr)

'H'

In [5]:
next(itr)

'i'

In [6]:
next(itr)

StopIteration: 

Also machen wir eigentlich Folgendes:

In [7]:
l = [2**x for x in range(10)]

itr = iter(l)
while True:
    try:
        n = next(itr)
        print(n)
    except StopIteration:
        break

1
2
4
8
16
32
64
128
256
512


## 3. Implementierung von Iteratoren mit Klassen

In [8]:
class MyIterator:
    def __init__(self, max):
        self.max = max
        self.curr = 0
        
    # the following methods are required for iterator objects
    
    def __next__(self):
        if self.curr < self.max:
            ret = self.curr
            self.curr += 1
            return ret
        else:
            raise StopIteration()
            
    def __iter__(self):
        return self

In [9]:
it = MyIterator(10)

In [10]:
next(it)

0

In [11]:
it = MyIterator(10)
while True:
    try:
        print(next(it))
    except StopIteration:
        break

0
1
2
3
4
5
6
7
8
9


In [12]:
it = MyIterator(10)
for i in it:
    print(i)

0
1
2
3
4
5
6
7
8
9


Ein Iterator ist ein *Einmal-Verwendungsobjekt*! Das heißt, sobald wir ihn verwendet haben, um über Elemente zu iterieren, können wir die Iteration normalerweise nicht zurücksetzen oder „zurückspulen“. Iterierbare Objekte, die mehrfach durchlaufen werden können, liefern für jeden Durchlauf neue Iteratoren zurück.

In [13]:
l = ['a', 'b', 'c', 'd', 'e']
for _ in range(3):
    for x in l:
        print(x, end=' ')

a b c d e a b c d e a b c d e 

In [14]:
l = ['a', 'b', 'c', 'd', 'e']
for _ in range(3):
    it = iter(l) # we obtain and "use up" an iterator each loop!
    while True:
        try:
            x = next(it)
            print(x, end=' ')
        except StopIteration:
            break

a b c d e a b c d e a b c d e 

Für einen Containertyp müssen wir eine `__iter__`-Methode implementieren, die einen Iterator zurückgibt.

In [15]:
class ArrayList:
    def __init__(self):
        self.data = []
        
    def append(self, val):
        self.data.append(None)
        self.data[len(self.data)-1] = val
        
    def __iter__(self):
        class ArrayListIterator:
            def __init__(self, data):
                self.data = data
                self.idx = 0
                
            def __next__(self):
                if self.idx < len(self.data):
                    ret = self.data[self.idx]
                    self.idx += 1
                    return ret
                else:
                    raise StopIteration()
            
            def __iter__(self):
                return self
            
        return ArrayListIterator(self.data)

In [16]:
l = ArrayList()
for x in range(10):
    l.append(2**x)

In [17]:
it = iter(l)

In [18]:
type(it)

__main__.ArrayList.__iter__.<locals>.ArrayListIterator

In [19]:
next(it)

1

In [20]:
for x in l:
    print(x)

1
2
4
8
16
32
64
128
256
512


## 4. Implementierung von Iteratoren mit Generatoren

Was ist ein „Generator“?

In [21]:
l = [2*x for x in range(10)]
g = (2*x for x in range(10))

In [22]:
type(l), type(g)

(list, generator)

In [23]:
for x in l:
    print(x)

0
2
4
6
8
10
12
14
16
18


In [24]:
for x in g:
    print(x)

0
2
4
6
8
10
12
14
16
18


In [25]:
dir(g)

['__class__',
 '__del__',
 '__delattr__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__iter__',
 '__le__',
 '__lt__',
 '__name__',
 '__ne__',
 '__new__',
 '__next__',
 '__qualname__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__str__',
 '__subclasshook__',
 'close',
 'gi_code',
 'gi_frame',
 'gi_running',
 'gi_suspended',
 'gi_yieldfrom',
 'send',
 'throw']

Normalerweise ist ein Generator deutlich leistungsfähiger als die Verwendung einer Liste:

In [26]:
%timeit -n 1000 [2*x for x in range(10_000)]

233 µs ± 1.63 µs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [27]:
%timeit -n 1000 (2*x for x in range(10_000))

367 ns ± 175 ns per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


Aber noch besser: Die Verwendung eines Generators benötigt viel weniger Speicher:

In [28]:
import sys
nums_squared_list = [i ** 2 for i in range(10000)]
print('Used memory list:', sys.getsizeof(nums_squared_list), 'bytes')
nums_squared_generator = (i ** 2 for i in range(10000))
print('Used memory generator:', sys.getsizeof(nums_squared_generator), 'bytes')

Used memory list: 85176 bytes
Used memory generator: 208 bytes


**???** Das ist ein deutlich geringerer Speicherverbrauch.

In [29]:
g = (2*x for x in range(10_000))

In [30]:
g[100]

TypeError: 'generator' object is not subscriptable

In [31]:
g[:100]

TypeError: 'generator' object is not subscriptable

In [32]:
sum(g)

99990000

In [33]:
sum(g)

0

Ein *Generatorausdruck* ähnelt syntaktisch einer Listenkomprehension und ist insofern ähnlich, als er zu einer iterierbaren Folge von Werten ausgewertet wird. Ein Generator stellt jedoch keine vollständig ausgearbeitete Sammlung von Werten dar; stattdessen werden Werte nur dann zurückgegeben, wenn sie über die Iterations-API (d.h. `next`) benötigt werden — dies bezeichnet man als *faule Auswertung*.

Dies macht einen Generator effizienter als eine Liste (da wir nicht alle Werte der Sequenz speichern müssen), aber Generatoren können Listen nicht in allen Szenarien ersetzen (z.B. wenn wir in der Sequenz springen oder Werte erneut aufrufen müssen).

### Erstellen von Generatorfunktionen: `yield`

In [34]:
def foo():
    yield

In [35]:
foo()

<generator object foo at 0x0000027AF2E7D850>

In [36]:
type(foo())

generator

In [37]:
def foo():
    print('hello!')
    yield
    print('goodbye!')

In [38]:
foo()

<generator object foo at 0x0000027AF2D9CA00>

In [39]:
g = foo()

In [40]:
next(g)

hello!


In [41]:
def foo():
    yield 1
    yield 2
    yield 3

In [42]:
g = foo()

In [43]:
next(g)

1

In [50]:
def countdown(n):
    for i in range(n, 0, -1):
        yield i
    yield "💣Boom💣"

In [51]:
for x in countdown(5):
    print(x)

5
4
3
2
1
💣Boom💣


In [52]:
list(countdown(10))

[10, 9, 8, 7, 6, 5, 4, 3, 2, 1, '💣Boom💣']

Eine *Generatorfunktion* ist eine Funktion, die eine oder mehrere `yield`-Anweisungen enthält. Wird sie aufgerufen, gibt eine Generatorfunktion ein Generatorobjekt zurück, das es uns ermöglicht, die Funktion schrittweise über die Iterations-API auszuführen. Jeder Aufruf von `next` am Generator führt die Funktion bis zur nächsten `yield`-Anweisung aus; wenn die Funktion beendet ist, löst der Generator eine `StopIteration`-Ausnahme aus (genau wie ein Iterator).

### Generatoren als Datenstruktur-Iteratoren

In [53]:
class ArrayList:
    def __init__(self):
        self.data = []
        
    def append(self, val):
        self.data.append(None)
        self.data[len(self.data)-1] = val
        
    def __iter__(self):
        for i in range(len(self.data)):
            yield self.data[i]

In [54]:
l = ArrayList()
for x in range(10):
    l.append(2**x)

In [55]:
for x in l:
    print(x)

1
2
4
8
16
32
64
128
256
512


In [60]:
class ArrayList(ArrayList):
    def __repr__(self):
        return '[' + ', '.join(str(x) for x in self) + ']'

In [61]:
l = ArrayList()
for x in range(10):
    l.append(2**x)
l

[1, 2, 4, 8, 16, 32, 64, 128, 256, 512]